# Comparing two runs

A single run is an anecdote. The platform exists to compare runs that differ in one
knob, which is what this notebook does: two runs, one changed field, the same
numbers read off both.

The knob here is `max_round_duration_seconds`, chosen because its effect is visible
without a real model. Cut a round short and the agents get less done in it, which
moves both the throughput number and the reason rounds ended.

In [ ]:
import os
import tempfile
from pathlib import Path

from pytest import MonkeyPatch

# These notebooks generate their own run, so nothing here reaches a provider.
# Clearing the keys makes that a fact rather than a claim: if any cell below
# tried to call a model, it would fail here rather than spend.
for _key in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"):
    os.environ.pop(_key, None)

SCENARIO = "warehouse_robot_recovery"
PRESET = "knobs_default"

In [ ]:
from glossogen.testing import run_rounds


async def generate_run(round_count, overrides):
    """Run the real round loop with the model replaced by a script.

    Every part of the platform is real here: the MCP server, the tool dispatch,
    the game clock, the world and the event logger. Only the LLM is scripted, so
    this costs nothing, needs no key, and gives the same answer every time.
    """
    with MonkeyPatch.context() as patch:
        return await run_rounds(
            scenario_name=SCENARIO,
            preset_name=PRESET,
            round_count=round_count,
            overrides=overrides,
            tmp_path=Path(tempfile.mkdtemp()),
            monkeypatch=patch,
        )

## Two runs, one difference

Everything else is held: the same preset, the same round count, the same scripted
agents.

In [ ]:
CONDITIONS = {
    "generous": {},
    "hurried": {"max_round_duration_seconds": 1},
}

runs = {}
for label, overrides in CONDITIONS.items():
    runs[label] = await generate_run(round_count=4, overrides=overrides)
    print(f"{label:10} {len(runs[label].events):5} events")

## Score both the same way

The same metric list over both runs. Reading them through the real evaluation
runner rather than by hand is what keeps this comparable to what `glossogen
evaluate` reports.

In [ ]:
from glossogen.evaluation.metric_core.metric_run_options import MetricRunOptions
from glossogen.testing import MetricRun, score_metrics

METRICS = [
    "round_success",
    "mean_chars_per_round",
    "mean_chars_per_message",
    "round_ended_idle",
    "round_ended_timeout",
]


async def score(simulation, label):
    """Score one run and return its measurements keyed by metric name."""
    run_dir = simulation.log_path.parent
    run = MetricRun(
        scenario=simulation.scenario,
        run_dir=run_dir,
        log_path=simulation.log_path,
        simulation=simulation,
    )
    with MonkeyPatch.context() as patch:
        scored = await score_metrics(
            run=run,
            metric_names=METRICS,
            judge_responses=[],
            options=MetricRunOptions(probe_round=None, probe_replicas=1, ontology_path=None),
            report_path=run_dir / f"report_{label}.json",
            monkeypatch=patch,
        )
    return {m.metric_name: m.score for m in scored.report.measurements}


scores = {}
for label, simulation in runs.items():
    scores[label] = await score(simulation, label)

In [ ]:
import pandas as pd

table = pd.DataFrame(scores).T
table.index.name = "condition"
table

## Read the difference, and check it is the one you asked for

The column to look at first is not the headline. It is `round_ended_timeout`: it
says whether the knob did what you meant. If the hurried condition ends its rounds
on the clock and the generous one does not, the manipulation worked and the
throughput difference is about the agents having less time. If both look the same,
the knob did not bite and any difference elsewhere is noise.

In [ ]:
for metric in ("round_ended_timeout", "round_ended_idle", "mean_chars_per_round"):
    generous = table.loc["generous", metric]
    hurried = table.loc["hurried", metric]
    print(f"{metric:24} generous={generous:7.2f}  hurried={hurried:7.2f}  "
          f"delta={hurried - generous:+7.2f}")

In [ ]:
import matplotlib.pyplot as plt

shown = ["mean_chars_per_round", "mean_chars_per_message"]
figure, axis = plt.subplots(figsize=(6, 3))
positions = range(len(shown))
width = 0.38
axis.bar([p - width / 2 for p in positions], [table.loc["generous", m] for m in shown],
         width, label="generous", color="#0e6b5c")
axis.bar([p + width / 2 for p in positions], [table.loc["hurried", m] for m in shown],
         width, label="hurried", color="#8d550e")
axis.set_xticks(list(positions))
axis.set_xticklabels(shown, fontsize=8)
axis.set_ylabel("characters")
axis.set_title("Throughput under a shortened round")
axis.legend()
figure.tight_layout()

## What this is not

Two runs with scripted agents differ only in what the platform did with a fixed
script, so this is a mechanism check rather than a result. A real comparison needs
replications at a fixed seed, because the same configuration run twice against a
real model gives different numbers, and the spread is often wider than the effect.

`seed=42` is the convention here for exactly that reason: it fixes the case set, so
repeated runs measure model variance on an identical workload rather than a
different workload each time.

For the real thing, see [Running simulations](../docs/running-simulations.md) on
sweeps, and [Evaluation](../docs/evaluation.md) on which metrics answer which
question.